In [1]:
import pandas as pd
import requests
import zipfile
import io
import os
import datetime
from sqlalchemy import create_engine, text, inspect
from urllib.parse import quote_plus
from dotenv import load_dotenv

In [2]:
# Carrega variáveis de ambiente
load_dotenv()

True

In [3]:
class CVMLakeBuilder:
    """
    Classe responsável pela orquestração do pipeline ETL de dados da CVM.
    Suporta ingestão de múltiplos tipos de documentos (FRE, DFP, ITR, CAD).
    """

    # Mapeamento de configurações por tipo de arquivo
    DATASET_CONFIG = {
        'FRE': {
            'base_url': 'https://dados.cvm.gov.br/dados/CIA_ABERTA/DOC/FRE/DADOS/',
            'file_prefix': 'fre_cia_aberta',
            'is_zip': True,
            'is_historical': True # Tem arquivos por ano (2010, 2011...)
        },
        'DFP': {
            'base_url': 'https://dados.cvm.gov.br/dados/CIA_ABERTA/DOC/DFP/DADOS/',
            'file_prefix': 'dfp_cia_aberta',
            'is_zip': True,
            'is_historical': True
        },
        'ITR': {
            'base_url': 'https://dados.cvm.gov.br/dados/CIA_ABERTA/DOC/ITR/DADOS/',
            'file_prefix': 'itr_cia_aberta',
            'is_zip': True,
            'is_historical': True
        },
        # CONFIGURAÇÃO DO CAD AJUSTADA
        'CAD': {
            'base_url': 'https://dados.cvm.gov.br/dados/CIA_ABERTA/CAD/DADOS/',
            'file_prefix': 'cad_cia_aberta', # Nome do arquivo sem extensão
            'is_zip': False,       # É um CSV direto
            'is_historical': False # É um arquivo único, não tem ano no nome
        }
    }

    def __init__(self, dataset_type, schema='layer_01_bronze'):
        self.dataset_type = dataset_type.upper()
        self.schema = schema
        
        if self.dataset_type not in self.DATASET_CONFIG:
            raise ValueError(f"Dataset '{dataset_type}' não configurado. Opções: {list(self.DATASET_CONFIG.keys())}")
            
        self.config = self.DATASET_CONFIG[self.dataset_type]
        self.log_table = f"{self.dataset_type.lower()}_raw_logs_by_timestamp"
        self.engine = self._create_db_engine()
        
        self.headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
        }

    def _create_db_engine(self):
        user = quote_plus(os.getenv('DB_USER', 'postgres'))
        password = quote_plus(os.getenv('DB_PASS', 'password'))
        host = os.getenv('DB_HOST', 'localhost')
        port = os.getenv('DB_PORT', '5432')
        dbname = os.getenv('DB_NAME', 'data_lake')
        
        connection_str = f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{dbname}"
        return create_engine(connection_str)

    def setup_database(self):
        create_log_query = f"""
        CREATE TABLE IF NOT EXISTS {self.schema}.{self.log_table} (
            log_id SERIAL PRIMARY KEY,
            data_execucao TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
            nivel_log VARCHAR(50),
            ano_referencia INT,
            arquivo_origem VARCHAR(200),
            tabela_destino VARCHAR(200),
            mensagem TEXT,
            schema_drift_detectado BOOLEAN DEFAULT FALSE
        );
        """
        try:
            with self.engine.connect() as conn:
                conn.execute(text(f"CREATE SCHEMA IF NOT EXISTS {self.schema};"))
                conn.execute(text(create_log_query))
                conn.commit()
            print(f"✅ Setup: Tabela de logs '{self.log_table}' verificada.")
        except Exception as e:
            print(f"❌ Erro no setup do banco: {e}")

    def _log(self, nivel, ano, arquivo, tabela, msg, drift=False):
        query = text(f"""
            INSERT INTO {self.schema}.{self.log_table} 
            (nivel_log, ano_referencia, arquivo_origem, tabela_destino, mensagem, schema_drift_detectado, data_execucao)
            VALUES (:niv, :ano, :arq, :tab, :msg, :drift, NOW())
        """)
        
        icon = "🟢" if nivel == 'SUCCESS' else "🟡" if nivel == 'WARNING' else "🔴"
        # Ajuste visual para quando não tem ano (CAD)
        ano_str = str(ano) if ano > 0 else "ATUAL"
        print(f"{icon} [{self.dataset_type}][{ano_str}][{nivel}] {msg}")

        try:
            with self.engine.connect() as conn:
                conn.execute(query, {'niv': nivel, 'ano': ano, 'arq': arquivo, 'tab': tabela, 'msg': msg, 'drift': drift})
                conn.commit()
        except Exception as e:
            print(f"Erro fatal ao tentar logar no banco: {e}")

    def _check_already_processed(self, filename):
        # Para arquivos únicos como CAD, talvez queiramos processar sempre (sobrescrever/append)
        # ou verificar se processamos hoje. Por enquanto, mantemos a lógica padrão.
        query = text(f"""
            SELECT 1 FROM {self.schema}.{self.log_table}
            WHERE arquivo_origem = :arq 
              AND nivel_log = 'SUCCESS'
              AND mensagem LIKE 'Processamento do arquivo finalizado%'
              -- Se for CAD, podemos querer verificar se já rodou HOJE, mas deixarei simples por ora
        """)
        with self.engine.connect() as conn:
            return conn.execute(query, {'arq': filename}).fetchone() is not None

    def _check_schema_drift(self, table_name, df_new):
        inspector = inspect(self.engine)
        if not inspector.has_table(table_name, schema=self.schema):
            return False, "Nova Tabela"

        cols_db = {col['name'] for col in inspector.get_columns(table_name, schema=self.schema)}
        cols_new = set(df_new.columns)
        new_cols = cols_new - cols_db
        
        if new_cols:
            return True, f"Drift: Novas colunas detectadas: {new_cols}"
        return False, "Schema OK"

    def _process_csv(self, csv_name, file_content, year, origin_file_name):
        """
        Lê o CSV (do zip ou direto), normaliza e salva.
        """
        try:
            # 1. Definição do nome da tabela
            # Limpeza do nome: remove extensão e ano se houver
            clean_name = csv_name.lower().replace('.csv', '')
            if year > 0:
                clean_name = clean_name.replace(f'_{year}', '')
            
            prefix = self.config['file_prefix'].split('_')[0] 
            
            if not clean_name.startswith(prefix):
                table_name = f"{prefix}_{clean_name}"
            else:
                table_name = clean_name
            
            # 2. Leitura do CSV
            df = pd.read_csv(
                file_content, 
                sep=';', 
                encoding='windows-1252', 
                on_bad_lines='skip', 
                low_memory=False
            )
            
            # 3. Metadados
            df['metadata_data_carga'] = datetime.datetime.now()
            df['metadata_arquivo_origem'] = origin_file_name
            
            df = df.astype(str)

            # 4. Drift e Carga
            is_drift, msg_drift = self._check_schema_drift(table_name, df)
            if is_drift:
                self._log('WARNING', year, csv_name, table_name, msg_drift, drift=True)

            df.to_sql(
                table_name,
                self.engine,
                schema=self.schema,
                if_exists='append',
                index=False,
                method='multi',
                chunksize=1000
            )
            
            self._log('INFO', year, csv_name, table_name, f"Carga OK: {len(df)} linhas.")

        except Exception as e:
            self._log('ERROR', year, csv_name, table_name if 'table_name' in locals() else 'unknown', f"Erro processando CSV: {str(e)}")

    def _limpar_recarga(self, ano, dry_run=False):
        """
        Remove os dados que serao recarregados, evitando duplicacao (if_exists='append').

        - Datasets historicos (FRE/DFP/ITR): remove apenas as linhas do ANO informado
          (identificadas pelo sufixo _{ano}.csv em metadata_arquivo_origem).
        - Dataset snapshot (CAD): remove TODAS as linhas do arquivo unico, pois e uma
          'foto' atual que substitui a anterior por completo.

        Seguranca: nunca usa DROP; nunca apaga a tabela de logs; so mexe nas tabelas
        deste dataset (mesmo prefixo). Com dry_run=True nada e apagado, apenas contado.
        Retorna o total de linhas afetadas.
        """
        inspector = inspect(self.engine)
        tabelas = inspector.get_table_names(schema=self.schema)
        prefix = self.config['file_prefix'].split('_')[0]  # ex: 'fre', 'dfp', 'itr', 'cad'

        if self.config['is_historical']:
            padrao = f'%{ano}.csv'
            escopo = f"ano {ano}"
        else:
            # CAD: arquivo unico sem ano -> remove o snapshot inteiro daquele arquivo
            padrao = f"{self.config['file_prefix']}%"
            escopo = "snapshot completo"

        total = 0
        modo = "🔍 SIMULACAO (dry_run)" if dry_run else "🧹 LIMPEZA"
        print(f"{modo} [{self.dataset_type}]: procurando dados ({escopo}) no schema {self.schema}...")

        with self.engine.connect() as conn:
            for tabela in tabelas:
                if tabela == self.log_table:
                    continue
                if not tabela.startswith(prefix):  # so tabelas deste dataset
                    continue
                cols = [c['name'] for c in inspector.get_columns(tabela, schema=self.schema)]
                if 'metadata_arquivo_origem' not in cols:
                    continue

                qtd = conn.execute(
                    text(f"SELECT COUNT(*) FROM {self.schema}.{tabela} "
                         f"WHERE metadata_arquivo_origem LIKE :pat"),
                    {'pat': padrao}
                ).scalar()

                if qtd:
                    sufixo = "(seriam removidas)" if dry_run else "removida(s)"
                    print(f"   - {tabela}: {qtd} linha(s) {sufixo}")
                    total += qtd
                    if not dry_run:
                        conn.execute(
                            text(f"DELETE FROM {self.schema}.{tabela} "
                                 f"WHERE metadata_arquivo_origem LIKE :pat"),
                            {'pat': padrao}
                        )

            if not dry_run:
                # Libera a trava de idempotencia para este dataset
                if self.config['is_historical']:
                    conn.execute(
                        text(f"DELETE FROM {self.schema}.{self.log_table} "
                             f"WHERE ano_referencia = :ano"),
                        {'ano': ano}
                    )
                else:
                    conn.execute(text(f"DELETE FROM {self.schema}.{self.log_table}"))
                conn.commit()  # tudo num commit so (rollback automatico em caso de erro)

        acao = "seriam removidas" if dry_run else "removidas"
        print(f"📊 Total [{self.dataset_type}]: {total} linha(s) {acao}.\n")
        return total

    def run_pipeline(self, start_year=2010, end_year=None):
        """
        Executa o pipeline. Se for histórico, usa o loop de anos. Se não (CAD), baixa o arquivo único.
        """
        if end_year is None:
            end_year = datetime.datetime.now().year

        self.setup_database()
        
        # Lógica para definir quais arquivos baixar
        files_to_process = []

        if self.config['is_historical']:
            print(f"\n🚀 Iniciando Pipeline Histórico ({start_year}-{end_year})...\n")
            for year in range(start_year, end_year + 1):
                file_name = f"{self.config['file_prefix']}_{year}.zip"
                url = f"{self.config['base_url']}{file_name}"
                files_to_process.append({'year': year, 'url': url, 'filename': file_name})
        else:
            # Lógica para arquivo único (CAD)
            print(f"\n🚀 Iniciando Pipeline Arquivo Único (Snapshot Atual)...\n")
            ext = ".zip" if self.config['is_zip'] else ".csv"
            file_name = f"{self.config['file_prefix']}{ext}"
            url = f"{self.config['base_url']}{file_name}"
            # Usamos ano=0 ou ano atual para representar o snapshot
            files_to_process.append({'year': end_year, 'url': url, 'filename': file_name})

        # Loop de processamento unificado
        for item in files_to_process:
            year = item['year']
            file_name = item['filename']
            url = item['url']

            # Idempotência:
            # - Datasets históricos: pula anos antigos já carregados, mas SEMPRE
            #   reprocessa o ano corrente (a CVM o atualiza continuamente).
            # - CAD (snapshot): sempre reprocessa, pois é a foto atual.
            eh_ano_atual = (year == end_year)
            forcar = eh_ano_atual or not self.config['is_historical']

            if self._check_already_processed(file_name) and not forcar:
                print(f"⏭️  {file_name} já processado. Pulando.")
                continue

            # Limpa os dados antigos antes de recarregar, evitando duplicação.
            if forcar:
                print(f"🔄 Atualizando para os dados mais recentes: {file_name}")
                self._limpar_recarga(year)

            try:
                self._log('INFO', year, file_name, '-', 'Baixando arquivo...')
                response = requests.get(url, headers=self.headers, timeout=120)
                
                if response.status_code != 200:
                    self._log('ERROR', year, file_name, '-', f"HTTP Erro: {response.status_code} - URL: {url}")
                    continue

                # DECISÃO: É ZIP ou CSV Direto?
                if self.config['is_zip']:
                    # Processamento ZIP (FRE, DFP, ITR)
                    try:
                        with zipfile.ZipFile(io.BytesIO(response.content)) as z:
                            csv_files = [f for f in z.namelist() if f.endswith('.csv')]
                            if not csv_files:
                                self._log('WARNING', year, file_name, '-', 'Zip vazio.')
                                continue
                            for csv in csv_files:
                                with z.open(csv) as f:
                                    self._process_csv(csv, f, year, file_name)
                    except zipfile.BadZipFile:
                        self._log('ERROR', year, file_name, '-', 'Arquivo corrompido ou não é um ZIP válido.')
                        continue
                else:
                    # Processamento CSV Direto (CAD)
                    # Envolvemos o conteúdo num BytesIO para simular um arquivo aberto
                    file_buffer = io.BytesIO(response.content)
                    self._process_csv(file_name, file_buffer, year, file_name)

                self._log('SUCCESS', year, file_name, 'TODAS', 'Processamento do arquivo finalizado.')

            except Exception as e:
                self._log('FATAL', year, file_name, '-', f"Erro crítico: {str(e)}")

In [8]:
# ==============================================================================
# EXECUÇÃO — roda TODOS os tipos de uma vez, sem precisar editar nada manualmente
#
# TIPOS   = lista de datasets a coletar. Comente o que não quiser.
#           (FRE já é tratado pelo notebook 2; deixei fora para não duplicar esforço.)
# DRY_RUN = True  -> apenas SIMULA cada tipo (mostra o que seria removido, não apaga
#                    nem baixa). Use para conferir antes.
#           False -> EXECUTA de verdade (limpa + recarrega os dados mais recentes).
#
# Fluxo recomendado: rode 1x com True, confira, troque para False e rode de novo.
# ==============================================================================
if __name__ == "__main__":
    TIPOS   = ['CAD', 'DFP', 'ITR']
    DRY_RUN = False

    ano_atual = datetime.datetime.now().year

    for tipo in TIPOS:
        print("=" * 70)
        print(f"  DATASET: {tipo}   ({'SIMULAÇÃO' if DRY_RUN else 'EXECUÇÃO REAL'})")
        print("=" * 70)

        pipeline = CVMLakeBuilder(dataset_type=tipo)

        if DRY_RUN:
            pipeline.setup_database()                       # garante schema/log p/ a contagem
            pipeline._limpar_recarga(ano_atual, dry_run=True)
        else:
            pipeline.run_pipeline(start_year=2010, end_year=ano_atual)

    print("✅ Fim." + ("  (nada foi alterado — modo simulação)" if DRY_RUN else ""))


  DATASET: CAD   (EXECUÇÃO REAL)
✅ Setup: Tabela de logs 'cad_raw_logs_by_timestamp' verificada.

🚀 Iniciando Pipeline Arquivo Único (Snapshot Atual)...

🔄 Atualizando para os dados mais recentes: cad_cia_aberta.csv
🧹 LIMPEZA [CAD]: procurando dados (snapshot completo) no schema layer_01_bronze...
   - cad_cia_aberta: 2650 linha(s) removida(s)
📊 Total [CAD]: 2650 linha(s) removidas.

🔴 [CAD][2026][INFO] Baixando arquivo...
🔴 [CAD][2026][INFO] Carga OK: 2675 linhas.
🟢 [CAD][2026][SUCCESS] Processamento do arquivo finalizado.
  DATASET: DFP   (EXECUÇÃO REAL)
✅ Setup: Tabela de logs 'dfp_raw_logs_by_timestamp' verificada.

🚀 Iniciando Pipeline Histórico (2010-2026)...

⏭️  dfp_cia_aberta_2010.zip já processado. Pulando.
⏭️  dfp_cia_aberta_2011.zip já processado. Pulando.
⏭️  dfp_cia_aberta_2012.zip já processado. Pulando.
⏭️  dfp_cia_aberta_2013.zip já processado. Pulando.
⏭️  dfp_cia_aberta_2014.zip já processado. Pulando.
⏭️  dfp_cia_aberta_2015.zip já processado. Pulando.
⏭️  dfp_cia_a